In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import matplotlib.pyplot as plt
import h5py
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dlai-happy-house/test.h5
/kaggle/input/dlai-happy-house/train.h5


In [2]:
train = h5py.File("/kaggle/input/dlai-happy-house/train.h5", "r")
test = h5py.File("/kaggle/input/dlai-happy-house/test.h5", "r")

In [3]:
test.keys()

<KeysViewHDF5 ['list_classes', 'test_set_x', 'test_set_y']>

In [73]:
def load(path, name):
    file_path = path
    with h5py.File(file_path, 'r') as f:
        data = f[name][:]  
    num_samples = data.shape[0]
    # flattened_data = data.reshape(num_samples, -1)
    
    df = pd.DataFrame([[data]])
    return df



In [96]:
X_train = load("/kaggle/input/dlai-happy-house/train.h5", "train_set_x")
Y_train = load("/kaggle/input/dlai-happy-house/train.h5", "train_set_y")
X_test = load("/kaggle/input/dlai-happy-house/test.h5", "test_set_x")
Y_test = load("/kaggle/input/dlai-happy-house/test.h5", "test_set_y")

In [97]:
X_train

,0
0,"[[[[178 190 163], [172 181 173], [188 196 184]..."


In [98]:
X_train = X_train[0][0]/255.
X_test = X_test[0][0]/255.

# Reshape
Y_train = Y_train[0][0].T
Y_test = Y_test[0][0].T
print ("number of training examples = " + str(X_train.shape[0]))
print ("number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))

number of training examples = 600
number of test examples = 150
X_train shape: (600, 64, 64, 3)
Y_train shape: (600,)
X_test shape: (150, 64, 64, 3)
Y_test shape: (150,)


In [41]:
type(Y_train)

tensorflow.python.data.ops.map_op._MapDataset

In [ ]:
# def normalize(image):
#     image = tf.cast(image, tf.float32)/255.0
#     image = tf.reshape(image, [-1,])
#     return image
# def transpose_element(element): 
#     return tf.transpose(element)
# X_train = X_train.map(normalize)
# X_test = X_test.map(normalize)

# Y_train = Y_train.map(transpose_element)
# Y_test = Y_test.map(transpose_element)

In [10]:
def happyModel():

    model = tf.keras.Sequential([
        tf.keras.layers.ZeroPadding2D(padding = 3, input_shape=(64,64,3)),
        tf.keras.layers.Conv2D(32,7,1),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPool2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1, activation = "sigmoid")
    ])

    return model

In [13]:
happy_model = happyModel()


In [14]:
happy_model.compile(optimizer = 'adam',
                    loss = 'binary_crossentropy',
                    metrics = ['accuracy'])

In [15]:
happy_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ zero_padding2d_2 (ZeroPadding2D)     │ (None, 70, 70, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 64, 64, 32)          │           4,736 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 64, 64, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_2 (ReLU)                       │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 32, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 32768)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │          32,769 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 37,633 (147.00 KB)

 Trainable params: 37,569 (146.75 KB)

 Non-trainable params: 64 (256.00 B)

In [99]:
happy_model.fit(X_train, Y_train, epochs = 10, batch_size = 16)

Epoch 1/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - accuracy: 0.6484 - loss: 1.7562
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.8411 - loss: 0.3611
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.9470 - loss: 0.1277
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.9659 - loss: 0.0973
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.9486 - loss: 0.1295
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.9410 - loss: 0.1475
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.9730 - loss: 0.0805
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.9417 - loss: 0.1650
Epoch 9/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.9527 - loss: 0.1530
Epoch 10/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.8958 - loss: 0.2867


In [100]:
happy_model.evaluate(X_test, Y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5992 - loss: 1.6401


[1.7045544385910034, 0.5866666436195374]